# XGBoost OBS Inference Service — End-to-End Verification (health check → baseline inference → hot swap loop)

This notebook verifies the complete **train → deploy → inference → hot swap** pipeline.

## Sections

| Section | Content |
|---|---|
| §1 | Configuration (⚠️ must edit) |
| §2 | Train two model sets (old = baseline, new = used to verify the hot swap) |
| §3 | Health check `GET /health` |
| §4 | Baseline inference `POST /` |
| **§5** | **Hot swap verification (core)** |
| §6 | Troubleshooting |

## Prerequisites

- The inference service is running:
  - Local docker (OBS API mode): `docker run --rm -d -p 18081:8080 -e OBS_BUCKET=<your-bucket-name> -e AccessKeyID=<AK> -e SecretAccessKey=<SK> xgb-bc:obs-minimal-v5`
  - Cloud ModelArts: deployment complete, with `OBS_BUCKET`/`AccessKeyID`/`SecretAccessKey` environment variables configured
- Credentials: a ModelArts API Key (for cloud inference) + OBS AK/SK (for replacing the model)
  - Recommended: pass them in via environment variables (`MODELARTS_API_KEY`, `OBS_AK`, `OBS_SK`); otherwise enter them interactively at runtime


## 0. Dependencies


In [ ]:
import subprocess, sys
for pkg in ("xgboost", "scikit-learn", "pandas", "esdk-obs-python", "requests"):
    try:
        __import__(pkg.replace("-", "_") if pkg != "esdk-obs-python" else "obs")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
print("Dependencies ready")


## 1. Configuration (⚠️ Must Edit)

Replace `<your-service-address>` / `<service-ID>` / `<your-bucket-name>` with actual
values, or override them via the environment variables
`MODELARTS_OBS_INFER_URL` / `OBS_BUCKET`.


In [ ]:
import getpass
import os
import shutil
from pathlib import Path

import json
import time
import requests
import urllib3

# === Service address ===
# Cloud ModelArts: https://<your-service-address>/v2/infer/<service-ID>/
# Local docker:    http://127.0.0.1:18081/
INFER_URL = os.environ.get(
    "MODELARTS_OBS_INFER_URL",
    "https://<your-service-address>/v2/infer/<service-ID>/").strip().rstrip("/") + "/"

# === OBS configuration (for the cloud hot swap) ===
OBS_BUCKET   = os.environ.get("OBS_BUCKET", "<your-bucket-name>")
OBS_KEY      = "models/xgboost_breast_cancer.json"
OBS_ENDPOINT = "https://obs.cn-north-4.myhuaweicloud.com"

# === Local paths (relative to this notebook's directory) ===
SCRIPT_DIR       = Path.cwd()
REQUEST_PATH     = SCRIPT_DIR / "sample_request.json"
OLD_MODEL        = SCRIPT_DIR / "model_out" / "old" / "xgboost_breast_cancer.json"
NEW_MODEL        = SCRIPT_DIR / "model_out" / "new" / "xgboost_breast_cancer.json"
LOCAL_MOUNT      = SCRIPT_DIR / "model_mount"
LOCAL_MOUNT_FILE = LOCAL_MOUNT / "xgboost_breast_cancer.json"

TIMEOUT    = 30.0
VERIFY_TLS = False   # False for private deployments with self-signed certificates; can be True for public services
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# === Validation ===
assert "<" not in INFER_URL and "<" not in OBS_BUCKET, (
    "Please replace <your-service-address>/<service-ID>/<your-bucket-name> with actual values first "
    "(or set the environment variables MODELARTS_OBS_INFER_URL / OBS_BUCKET)")
assert REQUEST_PATH.exists(), f"Not found: {REQUEST_PATH}"

is_local = "127.0.0.1" in INFER_URL or "localhost" in INFER_URL

if is_local:
    AUTH_HEADERS = {"Content-Type": "application/json"}
else:
    API_KEY = os.environ.get("MODELARTS_API_KEY", "").strip()
    if not API_KEY:
        API_KEY = getpass.getpass("ModelArts API Key: ").strip()
    AUTH_HEADERS = {"Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json"}

body = json.loads(REQUEST_PATH.read_text(encoding="utf-8"))

print(f"URL    = {INFER_URL}")
print(f"MODE   = {'local docker' if is_local else 'cloud ModelArts'}")
print(f"OBS    = obs://{OBS_BUCKET}/{OBS_KEY}")


## 2. Train Two Model Sets

The two model sets use different hyperparameters and produce different predictions,
so the effect of the hot swap is clearly visible after the replacement.
If you have already run this step and `model_out/` already contains the artifacts,
this section can be skipped.


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

def train_and_save(params, random_state, output_path, label):
    """Train, evaluate, and save a model."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y)
    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", n_jobs=-1, random_state=random_state, **params)
    model.fit(X_train, y_train, verbose=False)

    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(output_path))
    print(f"  {label}: Accuracy={acc:.4f} AUC={auc:.4f} "
          f"-> {output_path} ({output_path.stat().st_size:,} bytes)")

if not (OLD_MODEL.exists() and NEW_MODEL.exists()):
    print("Training the old model (baseline)...")
    train_and_save(
        dict(n_estimators=100, max_depth=3, learning_rate=0.1,
             subsample=0.8, colsample_bytree=0.8),
        random_state=42, output_path=OLD_MODEL, label="OLD")
    print("Training the new model...")
    train_and_save(
        dict(n_estimators=250, max_depth=6, learning_rate=0.01,
             subsample=0.6, colsample_bytree=0.5, min_child_weight=5,
             reg_alpha=0.5, reg_lambda=2.0, gamma=0.5),
        random_state=2024, output_path=NEW_MODEL, label="NEW")
else:
    print(f"Models already exist, skipping training: {OLD_MODEL.stat().st_size:,}B / {NEW_MODEL.stat().st_size:,}B")


## 3. Health Check `GET /health`


In [ ]:
resp = requests.get(INFER_URL.rstrip("/") + "/health",
                    headers=AUTH_HEADERS, timeout=TIMEOUT, verify=VERIFY_TLS)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

h = resp.json()
assert h["status"] == "ok"
assert h["feature_count"] == 30
print(f"\nsync_mode    = {h['sync_mode']}")
print(f"model_origin = {h['model_origin']}")
if h["sync_mode"] == "obs-api":
    assert h["model_source"].startswith("obs://"), (
        f"obs-api mode but the model was not synced from OBS: {h.get('obs', {}).get('error')}")
print("\nHEALTH OK ✅")


## 4. Baseline Inference `POST /`


In [ ]:
resp = requests.post(INFER_URL, headers=AUTH_HEADERS, json=body,
                    timeout=TIMEOUT, verify=VERIFY_TLS)
resp.raise_for_status()
pred_baseline = float(resp.json()[0]["predictresult"])
print(f"predictresult = {pred_baseline:.16f}")

## 5. Hot Swap Verification (Core)

Replace the models one by one and verify that `predictresult` changes automatically.

**Replacement strategy**: delete first → then upload (so the OBS file system
notices the change).


In [ ]:
obs_client = None

if not is_local:
    # Cloud: replace the object via the OBS API (deleteObject → putFile, two steps)
    from obs import ObsClient
    ak = os.environ.get("OBS_AK", "") or getpass.getpass("OBS AK: ")
    sk = os.environ.get("OBS_SK", "") or getpass.getpass("OBS SK: ")
    obs_client = ObsClient(access_key_id=ak, secret_access_key=sk,
                           server=OBS_ENDPOINT, timeout=30)

def replace_model(src_path):
    """Replace the model: delete the old file/object first, then write the new one."""
    if is_local:
        # Local: delete → copy (requires the container to be started with -v model_mount:/opt/model)
        LOCAL_MOUNT.mkdir(parents=True, exist_ok=True)
        if LOCAL_MOUNT_FILE.exists():
            LOCAL_MOUNT_FILE.unlink()
            print("  [local] old file deleted")
        shutil.copy2(str(src_path), str(LOCAL_MOUNT_FILE))
        print(f"  [local] written: {src_path.name} ({src_path.stat().st_size:,} bytes)")
    else:
        resp = obs_client.deleteObject(OBS_BUCKET, OBS_KEY)
        print(f"  [OBS] delete status={resp.status}: obs://{OBS_BUCKET}/{OBS_KEY}")
        resp = obs_client.putFile(OBS_BUCKET, OBS_KEY, str(src_path))
        assert resp.status < 300, (
            f"putFile failed: status={resp.status}, "
            f"errorCode={getattr(resp, 'errorCode', '')}, "
            f"errorMessage={getattr(resp, 'errorMessage', '')}")
        print(f"  [OBS] uploaded: {src_path.name} ({src_path.stat().st_size:,} bytes)")

def infer_once(label=""):
    resp = requests.post(INFER_URL, headers=AUTH_HEADERS, json=body,
                         timeout=TIMEOUT, verify=VERIFY_TLS)
    resp.raise_for_status()
    p = float(resp.json()[0]["predictresult"])
    tag = f" [{label}]" if label else ""
    print(f"  predictresult{tag} = {p:.16f}")
    return p

print("Initialization complete ✅")


In [ ]:
# Step 1: replace with the old model
print("=" * 55)
print("Replacing with the [OLD] model")
print("=" * 55)
replace_model(OLD_MODEL)
time.sleep(1 if is_local else 3)
pred_old = infer_once("old model")


In [ ]:
# Step 2: replace with the new model
print("=" * 55)
print("Replacing with the [NEW] model")
print("=" * 55)
replace_model(NEW_MODEL)
time.sleep(1 if is_local else 3)
pred_new = infer_once("new model")


In [ ]:
# Step 3: compare the results
diff = abs(pred_new - pred_old)
print("=" * 55)
print(f"  Old model: {pred_old:.16f}")
print(f"  New model: {pred_new:.16f}")
print(f"  Diff:      {diff:.10f}")
print("=" * 55)

if diff > 1e-6:
    print("\n  ✅ Hot swap succeeded! predictresult changed automatically without restarting the service")
else:
    print("\n  ⚠️ Hot swap did not take effect")
    raise AssertionError(f"diff={diff}")

if obs_client:
    obs_client.close()


## 6. Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Cloud hot swap doesn't take effect after replacing the OBS object | The OBS file system doesn't refresh mtime | Stick to the two-step "deleteObject first, then putFile" (this notebook already implements it) |
| Local docker hot swap doesn't take effect | The model_mount directory isn't mounted with `-v` | Recreate the container with `-v model_mount:/opt/model` added |
| `/health`'s `model_source` doesn't start with `obs://` | Never entered OBS API mode (`OBS_BUCKET`/AK/SK not all configured) | Check `sync_mode_reason` in the response, supply whichever is missing, and restart |
| `[obs-probe] FAILED status=403` | AK/SK invalid or no permission on the bucket | Check the credentials and bucket policy; the service degrades to the baked-in fallback model and keeps running |
| Inference 401/403 | Invalid API Key | Check the `Authorization: Bearer <Token>` request header |
